In [67]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import tqdm
from tqdm.auto import tqdm

In [68]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

random.shuffle(words)
print(words[:8])
len(words)

['mazzy', 'foster', 'perry', 'mekenzie', 'ash', 'mahira', 'dianey', 'emmersyn']


32033

In [69]:
if (device := torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")):
    print(f"Using device: {device}")

block_size = 16
epochs = 400
lr = 0.001

Using device: cuda


In [70]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [71]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0]
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [72]:
X, Y = build_dataset(words)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 1]) torch.Size([228146])
torch.Size([205331, 1]) torch.Size([205331])
torch.Size([22815, 1]) torch.Size([22815])


In [73]:
trainds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())
traindl = DataLoader(
    dataset=trainds,
    batch_size=512,
    pin_memory=False,
    shuffle=True,
    num_workers=16,
    prefetch_factor=16
)

In [74]:
class GRUCell(torch.nn.Module):
    def __init__(self, emd_size, hid_size):
        super().__init__()
        self.update = nn.Linear(emb_size + hid_size, hid_size)
        self.reset = nn.Linear(emb_size + hid_size, hid_size)
        self.candidate = nn.Linear(emb_size + hid_size, hid_size)
    def gate(self, x, h, layer, activation):
        xh = torch.cat([x, h], dim=1)
        return activation(layer(xh))
    def forward(self, x, h):
        z = self.gate(x, h, self.update, torch.sigmoid) # how much new memory to use
        r = self.gate(x, h, self.reset, torch.sigmoid) # how much old affects new
        h_tilde = self.gate(x, r * h, self.candidate, torch.tanh) # new memory guess

        h = (1-z) * h + z * h_tilde

In [77]:
class GRU(torch.nn.Module):
    def __init__(self, vocab_size, emb_size, hid_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.cell = GRUCell(emb_size, hid_size)
        self.head = nn.Linear(hid_size, vocab_size)
        self.n_hidden = hid_size
    def forward(self, X, h=None):
        B,T = X.shape

        if h is None:
            h = torch.zeros(B, self.n_hidden, device=X.device)

        emb = self.embedding(X)  # (B, T, emb_size)
        logits = []

        for t in range(T):
            x = emb[:, t, :]
            h = self.cell(x, h)
            logits_t = self.head(h)
            logits.append(logits_t)

        logits = torch.stack(logits, dim=1)  # (B, T, vocab_size)
        return logits, h


In [79]:
model = GRU(vocab_size=27, emb_size=10, hid_size=100).to(device)
# model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma = 0.99995)

NameError: name 'emb_size' is not defined

In [ ]:
model.train()
pbar = tqdm(range(epochs), desc="train")
for e in pbar:
    for Xb, Yb in traindl:
        Xb, Yb = Xb.to(device), Yb.to(device)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    # if e % 10 == 0:
    #     print(e, loss.item(), scheduler.get_last_lr()[0])